# Inverting objectives

Several MCDA methods internally combine criteria by *adding* or
*multiplying* their values (`WeightedSumModel`, `WeightedProductModel`,
the `MOORA` family, `WASPAS`, `CoCoSo`, `CODAS`, ...). Those operations
only make sense if **every criterion points in the same direction**
(higher is always better) and, for the multiplicative ones, if every
value is strictly positive. `skcriteria.preprocessing.invert_objectives`
provides several strategies to get there. This tutorial walks through
all of them and when to pick each one.

## Case

A city is choosing a power-plant technology among four candidates
(*T1* to *T4*), considering:

1. **Cost** (USD millions). Sense of optimality, $Minimize$.
2. **Efficiency** (%). Sense of optimality, $Maximize$.

In [1]:
import skcriteria as skc

dm = skc.mkdm(
    matrix=[
        [120, 38],
        [95, 42],
        [140, 55],
        [80, 30],
    ],
    objectives=[min, max],
    weights=[0.6, 0.4],
    alternatives=["T1", "T2", "T3", "T4"],
    criteria=["Cost", "Efficiency"],
)
dm

,Cost[▼ 0.6],Efficiency[▲ 0.4]
T1,120,38
T2,95,42
T3,140,55
T4,80,30


## 1. The problem: mixing MAX and MIN criteria

`Cost` should be minimized and `Efficiency` maximized. Methods that
combine criteria by addition or multiplication have no way to know that
a *lower* `Cost` is "better" — internally they just add up
`weight * value`, so a smaller `Cost` would wrongly *reduce* the score
instead of improving it. Because of that, `skcriteria` refuses to
evaluate a matrix with mixed objectives on those methods and raises an
explicit error instead of returning a silently wrong ranking:

In [2]:
from skcriteria.agg.simple import WeightedSumModel

try:
    WeightedSumModel().evaluate(dm)
except ValueError as err:
    print("ValueError:", err)

ValueError: WeightedSumModel can't operate with minimize objective


The fix is always the same idea: turn every `Minimize` criterion into an
equivalent `Maximize` one *before* calling the method. `skcriteria`
offers several ways to do this, each with different trade-offs.

## 2. Negating vs. inverting: `NegateMinimize` vs. `InvertMinimize`

- `NegateMinimize` replaces every value $x$ of a `Minimize` criterion by
  $-x$. It is a simple sign flip: $\min{C} \equiv \max{-C}$.
- `InvertMinimize` replaces every value $x$ by $1/x$:
  $\min{C} \equiv \max{1/C}$.

(`MinimizeToMaximize` is a **deprecated alias** of `InvertMinimize` kept
only for backwards compatibility — prefer `InvertMinimize` in new code.)

In [3]:
from skcriteria.preprocessing import invert_objectives

display(invert_objectives.NegateMinimize().transform(dm))
display(invert_objectives.InvertMinimize().transform(dm))

,Cost[▲ 0.6],Efficiency[▲ 0.4]
T1,-120.0,38
T2,-95.0,42
T3,-140.0,55
T4,-80.0,30


,Cost[▲ 0.6],Efficiency[▲ 0.4]
T1,0.008333,38
T2,0.010526,42
T3,0.007143,55
T4,0.012500,30


Both make `Cost` a `Maximize` criterion and preserve the *order* of the
alternatives (`T4` is still the cheapest, `T3` the most expensive), but
they behave very differently:

- `NegateMinimize` is **linear**: it just mirrors the scale around zero,
  so the *differences* between alternatives are preserved exactly.
  However it can (and, for any positive `Cost`, always will) produce
  **negative values**, which breaks any method that requires positive
  data (`WeightedProductModel`, the multiplicative members of the
  `MOORA` family, etc. — see Part 4).
- `InvertMinimize` is **non-linear** and always keeps the values
  **positive** (as long as no value is `0`), but it compresses large
  values much more aggressively than small ones. This is exactly the
  effect explored in the
  [Scaling and weighting criteria](scale_weight.ipynb) tutorial, where
  inverting *Price* without rescaling afterwards made it numerically
  irrelevant to `WeightedSumModel` despite being the most important
  criterion.

<div class="alert alert-info">

**Rule of thumb:** use `NegateMinimize` when the method tolerates
negative values and you want to preserve the original scale as closely
as possible (e.g. before a distance-based method like `TOPSIS`); use
`InvertMinimize` when the method needs strictly positive values and you
are going to rescale afterwards anyway.

</div>

## 3. `MinMaxInverter` and `BenefitCostInverter`

Both `MinMaxInverter` and `BenefitCostInverter` do two jobs at once:
invert the `Minimize` criteria **and** normalize every criterion to a
comparable, always-positive scale — so, unlike Part 2, there is no
separate scaling step needed afterwards.

- `MinMaxInverter` rescales every criterion to $[0, 1]$: benefit
  criteria with $(x - \min)/(\max - \min)$, cost criteria with
  $(x - \max)/(\min - \max)$.
- `BenefitCostInverter` uses ratios instead: benefit criteria with
  $x/\max$, cost criteria with $\min/x$.

In [4]:
mm = invert_objectives.MinMaxInverter().transform(dm)
bc = invert_objectives.BenefitCostInverter().transform(dm)

display(mm)
display(bc)

,Cost[▲ 0.6],Efficiency[▲ 0.4]
T1,0.333333,0.32
T2,0.750000,0.48
T3,-0.000000,1.00
T4,1.000000,0.00


,Cost[▲ 0.6],Efficiency[▲ 0.4]
T1,0.666667,0.690909
T2,0.842105,0.763636
T3,0.571429,1.000000
T4,1.000000,0.545455


Notice that `MinMaxInverter` maps the *worst* alternative of every
criterion to exactly `0` (`T3`'s `Cost` and `T4`'s `Efficiency`), while
`BenefitCostInverter` never touches `0` for positive data — the worst
alternative simply gets the smallest positive ratio. That literal `0` is
harmless for `WeightedSumModel`, but it breaks any method that divides
or takes a log of the matrix, such as `WeightedProductModel`:

In [5]:
from skcriteria.agg.simple import WeightedProductModel

try:
    WeightedProductModel().evaluate(mm)
except ValueError as err:
    print("MinMaxInverter -> ValueError:", err)

WeightedProductModel().evaluate(bc)

MinMaxInverter -> ValueError: WeightedProductModel can't operate with values <= 0


Alternatives,T1,T2,T3,T4
Rank,4,1,3,2


<div class="alert alert-warning">

**Note:** `BenefitCostInverter` also raises a `ValueError` if the
original matrix already contains **negative** values — it is designed
to work on raw, non-negative data. If your data has negatives, Part 4
shows how to fix that first.

</div>

## 4. Zeros and negatives: `AddValueToZero` and `PushNegatives`

Ratio- and product-based methods (`WeightedProductModel`,
`FullMultiplicativeForm`, `MultiMOORA`, ...) divide by criterion values
or take their logarithm, so a single `0` or negative value in the matrix
is enough to make the whole computation break (division by zero,
logarithm of a negative number, or a sign flip that silently reverses
the meaning of "better"). `skcriteria.preprocessing` offers two small,
composable transformers to clean this up:

- `AddValueToZero(target, value)`: adds `value` to a criterion **only if
  that criterion contains an exact `0`** — every other criterion is left
  untouched. It operates **per column**.
- `PushNegatives(target)`: if the matrix (or weights) contains any
  negative value, it shifts **the whole matrix by the same amount** (the
  absolute value of the global minimum), so everything becomes `>= 0`.
  It operates on the whole array at once, not per column.

In [6]:
from skcriteria.preprocessing import increment, push_negatives

toy = skc.mkdm(
    matrix=[[1, -1, 5], [2, 0, -3]],
    objectives=[max, max, max],
    criteria=["A", "B", "C"],
)
display(toy)
display(push_negatives.PushNegatives(target="matrix").transform(toy))
display(increment.AddValueToZero(target="matrix", value=1).transform(toy))

,A[▲ 1.0],B[▲ 1.0],C[▲ 1.0]
A0,1,-1,5
A1,2,0,-3


,A[▲ 1.0],B[▲ 1.0],C[▲ 1.0]
A0,4,2,8
A1,5,3,0


,A[▲ 1.0],B[▲ 1.0],C[▲ 1.0]
A0,1.0,0.0,5.0
A1,2.0,1.0,-3.0


`PushNegatives` shifted **every** column by `5` (the absolute value of
the global minimum, `-5` in criterion `C`) — even `A`, which had no
negative values to begin with. `AddValueToZero`, on the other hand, only
touched criterion `B` (the only one with a literal `0`) and left the
negative value in `C` untouched, since it doesn't deal with negatives at
all. The two transformers solve different problems and are usually
chained together, `PushNegatives` first.

## 5. Integrator exercise: preparing a "dirty" matrix for MOORA

Let's combine everything on a genuinely messy matrix — mixed
objectives, negative values *and* a zero — borrowed from the `SPOTIS`
literature <cite data-cite="dezert2020spotis">[dezert2020spotis]</cite>
(SPOTIS itself is the topic of a future tutorial; here we only reuse its
example matrix as a good stress test).

In [7]:
dirty_dm = skc.mkdm(
    matrix=[
        [10.5, -3.1, 1.7],
        [-4.7, 0.0, 3.4],
        [8.1, 0.3, 1.3],
        [3.2, 7.3, -5.3],
    ],
    objectives=[max, min, max],
    weights=[0.2, 0.3, 0.5],
    alternatives=["A1", "A2", "A3", "A4"],
    criteria=["C1", "C2", "C3"],
)
dirty_dm

,C1[▲ 0.2],C2[▼ 0.3],C3[▲ 0.5]
A1,10.5,-3.1,1.7
A2,-4.7,0.0,3.4
A3,8.1,0.3,1.3
A4,3.2,7.3,-5.3


`FullMultiplicativeForm` (one of the four `MOORA` variants, see the
upcoming aggregation-methods tutorial) is a ratio of products: it
multiplies every `Maximize` value and divides by the product of every
`Minimize` value, so it needs the whole matrix to be **strictly
positive**. Calling it directly fails immediately:

In [8]:
from skcriteria.agg.moora import FullMultiplicativeForm

try:
    FullMultiplicativeForm().evaluate(dirty_dm)
except ValueError as err:
    print("ValueError:", err)

ValueError: FullMultiplicativeForm can't operate with values <= 0


The cleaning pipeline follows directly from Parts 2-4:

1. `NegateMinimize` turns `C2` into a `Maximize` criterion (still
   possibly negative).
2. `PushNegatives` shifts the *whole* matrix so every value is `>= 0`.
3. `AddValueToZero` nudges away the exact `0` that `PushNegatives` may
   have left behind (`A2`'s `C2`, which was exactly at the global
   minimum).

In [9]:
step1 = invert_objectives.NegateMinimize().transform(dirty_dm)
step2 = push_negatives.PushNegatives(target="matrix").transform(step1)
step3 = increment.AddValueToZero(target="matrix", value=1.0).transform(step2)

display(step1)
display(step2)
display(step3)

,C1[▲ 0.2],C2[▲ 0.3],C3[▲ 0.5]
A1,10.5,3.1,1.7
A2,-4.7,-0.0,3.4
A3,8.1,-0.3,1.3
A4,3.2,-7.3,-5.3


,C1[▲ 0.2],C2[▲ 0.3],C3[▲ 0.5]
A1,17.8,10.4,9.0
A2,2.6,7.3,10.7
A3,15.4,7.0,8.6
A4,10.5,0.0,2.0


,C1[▲ 0.2],C2[▲ 0.3],C3[▲ 0.5]
A1,17.8,11.4,9.0
A2,2.6,8.3,10.7
A3,15.4,8.0,8.6
A4,10.5,1.0,2.0


In [10]:
FullMultiplicativeForm().evaluate(step3)

Alternatives,A1,A2,A3,A4
Rank,1,3,2,4


The same three-step recipe (`NegateMinimize` → `PushNegatives` →
`AddValueToZero`) works for any method that is sensitive to zeros or
negatives, and can be written as a single `pipeline`
([introduced in the Quick Start](quickstart.ipynb)) so it never has to
be done by hand again:

In [11]:
from skcriteria.pipelines import mkpipe

pipe = mkpipe(
    invert_objectives.NegateMinimize(),
    push_negatives.PushNegatives(target="matrix"),
    increment.AddValueToZero(target="matrix", value=1.0),
    FullMultiplicativeForm(),
)
pipe.evaluate(dirty_dm)

Alternatives,A1,A2,A3,A4
Rank,1,3,2,4
